# MuonClip: saved initialization vs saved final angular spectrum

This notebook loads two **actual** RG Optimizers checkpoints:

```text
checkpoint_initial.pt   # step 0
checkpoint_final.pt     # completed final step
```

It obtains all six one-head NanoGPT matrices through WeightWatcher
`get_weights`, computes the gauge-invariant tilt and twist angular ESDs, and
compares them with a randomized-initial null that preserves the initial singular
values. An apparent Pareto exponent near two is not evidence unless the actual
angular ESD and fitted exponent both differ from the null.

Variables must be exported **before Jupyter starts**:

```bash
cd /path/to/rg_optimizers
export RG_OPTIMIZERS_ROOT="$PWD"
export RUNROOT=/tmp/<same-run-root-used-by-training>
export RESULTS_ROOT="$RUNROOT/results"
export TARGET_OPTIMIZER=muon_clip
export TARGET_SEED=4242
export RUN_DIR="$RESULTS_ROOT/$TARGET_OPTIMIZER/seed_$TARGET_SEED"

jupyter lab baseline/nanogpt_one_head/notebooks/angular/07_muonclip_initial_final_angular_weightwatcher.ipynb
```

Resolution precedence is `RUN_DIR`, then `RESULTS_ROOT`, then `RUNROOT`, then
shallow discovery. For another seed, change only `TARGET_SEED`. Optional:
`INITIAL_CHECKPOINT_PATH`, `FINAL_CHECKPOINT_PATH`, `ANGULAR_OUTPUT_DIR`,
`ANGULAR_N_NULL`, `ANGULAR_N_ENTRY_NULL`, and `ANGULAR_SHOW_PLOTS`.


In [ ]:
from pathlib import Path
import os
import sys


def find_experiment_root() -> Path:
    candidates = []
    configured = os.environ.get("RG_OPTIMIZERS_ROOT")
    if configured:
        root = Path(configured).expanduser().resolve()
        candidates += [root, root / "baseline" / "nanogpt_one_head"]
    cwd = Path.cwd().resolve()
    for base in (cwd, *cwd.parents):
        candidates += [base, base / "baseline" / "nanogpt_one_head"]
    for candidate in dict.fromkeys(candidates):
        if (candidate / "src" / "rg_nanogpt_one_head" / "model.py").is_file():
            return candidate
    raise FileNotFoundError(
        "Set RG_OPTIMIZERS_ROOT or launch Jupyter from the rg_optimizers repository"
    )


EXPERIMENT_ROOT = find_experiment_root()
sys.path.insert(0, str(EXPERIMENT_ROOT / "src"))
print("EXPERIMENT_ROOT =", EXPERIMENT_ROOT)


In [ ]:
from IPython.display import display
from rg_nanogpt_one_head.angular_weightwatcher_core import AnalysisConfig
from rg_nanogpt_one_head.angular_weightwatcher_pipeline import run_analysis

CONFIG = AnalysisConfig.from_env()
print(CONFIG)


In [ ]:
RESULTS, RESOLVED_RUN, ANALYSIS_MANIFEST = run_analysis(CONFIG)
print("RUN_DIR =", RESOLVED_RUN.run_dir, "via", RESOLVED_RUN.run_dir_source)
print("INITIAL =", RESOLVED_RUN.initial_path)
print("FINAL =", RESOLVED_RUN.final_path)
print("OUTPUT =", RESOLVED_RUN.output_dir)
display(RESULTS)


## Interpretation

- Actual angular ESD inside the randomized-initial interval: random-compatible.
- Actual \(\alpha\approx2\) when the null interval also contains two: no
  power-law evidence.
- Nonrandom ESD but exponent inside the null interval: angular learning exists,
  but not a distinct scale-free tail.
- Nonrandom ESD, acceptable fit, and exponent outside the null: candidate
  learned angular power-law structure.

For square `W_Q`, `W_K`, `W_V`, and `W_O`, the tilt sector is structurally
trivial; interpret the twist sector. Inspect both sectors for rectangular MLP
matrices. Numerical tables, plots, and `analysis_manifest.json` are saved below
the selected run's `diagnostics/` directory unless `ANGULAR_OUTPUT_DIR` is set.
